In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from interpret.glassbox import ExplainableBoostingClassifier
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import PartialDependenceDisplay
import warnings
warnings.filterwarnings("ignore")

#page steup
st.set_page_config(page_title="Explainable Credit Approval Dashboard", layout="wide")
st.markdown("""
<style>
body {background-color: #f8f9fa;}
.main {background-color: #ffffff; padding: 25px; border-radius: 12px;}
h1, h2, h3, h4 {color: #2c3e50;}
.stButton>button {
    background-color: #4CAF50;
    color: white;
    border-radius: 10px;
    height: 3em;
    width: 10em;
    font-weight: bold;
    border: none;
    transition: 0.3s;
}
.stButton>button:hover {
    background-color: #45a049;
    transform: scale(1.05);
}
</style>
""", unsafe_allow_html=True)

st.title("🏦 Explainable Credit Approval Dashboard")
st.markdown("Upload datasets and run the complete ML pipeline — all steps from your notebook are preserved here, including correlation, explainability, and fairness.")

#file upload
col1, col2, col3 = st.columns(3)
with col1:
    internal_file = st.file_uploader("Internal Dataset (.xlsx)", type=["xlsx"])
with col2:
    external_file = st.file_uploader("External Dataset (.xlsx)", type=["xlsx"])
with col3:
    unseen_file = st.file_uploader("Unseen Dataset (.xlsx) - Optional", type=["xlsx"])

#help function
def find_common_key(a: pd.DataFrame, b: pd.DataFrame):
    candidates = ['ID','Id','id','customer_id','cust_id','Client_ID','CLIENT_ID','client_id','Account_ID','account_id','clientid']
    a_cols_lower = {col.lower(): col for col in a.columns}
    b_cols_lower = {col.lower(): col for col in b.columns}
    for lower_col, orig_col in a_cols_lower.items():
        if lower_col in b_cols_lower:
            return orig_col, b_cols_lower[lower_col]
    for cand in candidates:
        if cand.lower() in a_cols_lower and cand.lower() in b_cols_lower:
            return a_cols_lower[cand.lower()], b_cols_lower[cand.lower()]
    return None, None

#model
if st.button("Start Model"):
    if not internal_file or not external_file:
        st.error("Please upload both internal and external datasets first.")
    else:
        with st.spinner("Running your full ML pipeline... "):
            # Load data
            df_int = pd.read_excel(internal_file)
            df_ext = pd.read_excel(external_file)

            # Merge datasets
            key_a, key_b = find_common_key(df_int, df_ext)
            if key_a and key_b:
                st.write(f"Merging on detected key: `{key_a}`")
                df = pd.merge(df_int, df_ext, left_on=key_a, right_on=key_b, how='inner', suffixes=('_int','_ext'))
                if key_a != key_b and key_b in df.columns:
                    df.drop(columns=[key_b], inplace=True)
            else:
                if df_int.shape[0] == df_ext.shape[0]:
                    df = pd.concat([df_int.reset_index(drop=True), df_ext.reset_index(drop=True)], axis=1)
                else:
                    df = pd.concat([df_int, df_ext], axis=1)
            df = df.loc[:, ~df.columns.duplicated()]
            st.write("Merged dataset shape:", df.shape)

            # Target creation
            if 'Approved_Flag' in df.columns:
                target_col = 'Approved_Flag'
            elif 'Credit_Score' in df.columns:
                df['Approved_Flag_generated'] = (df['Credit_Score'] >= 600).astype(int)
                target_col = 'Approved_Flag_generated'
            elif 'num_times_delinquent' in df.columns:
                df['Approved_Flag_generated'] = (df['num_times_delinquent'] == 0).astype(int)
                target_col = 'Approved_Flag_generated'
            else:
                np.random.seed(42)
                df['Approved_Flag_generated'] = np.random.binomial(1, 0.7, size=df.shape[0])
                target_col = 'Approved_Flag_generated'

            df[target_col] = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)

            # Feature separation
            features = [c for c in df.columns if c not in [target_col, key_a, key_b] if c in df.columns]
            num_cols = df[features].select_dtypes(include=['int64','float64']).columns.tolist()
            cat_cols = df[features].select_dtypes(include=['object','category','bool']).columns.tolist()
            if 'GENDER' in df.columns and 'GENDER' in num_cols:
                num_cols.remove('GENDER')
                cat_cols.append('GENDER')

            # Correlation plot
            st.subheader("Correlation Heatmap")
            fig, ax = plt.subplots(figsize=(12,8))
            sns.heatmap(df[num_cols].corr(), cmap='coolwarm', ax=ax)
            st.pyplot(fig)

            # Synthetic target
            df['Approved_Flag_synth'] = (df['Credit_Score'] >= 650).astype(int)
            y = df['Approved_Flag_synth']
            X = df.drop(columns=['Approved_Flag_synth'])
            X = X.loc[:, ~X.columns.duplicated()]

            # Train-test split
            X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

            # Preprocessing
            preproc = ColumnTransformer([
                ('num', StandardScaler(), num_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
            ])

            # Logistic Regression
            st.subheader("Logistic Regression")
            lr = Pipeline([
                ('preproc', preproc),
                ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
            ])
            lr.fit(X_train, y_train)
            y_pred_lr = lr.predict(X_test)
            st.code(classification_report(y_test, y_pred_lr), language='text')

            # SHAP
            st.subheader("SHAP Feature Importance")
            X_test_trans = lr.named_steps['preproc'].transform(X_test)
            explainer = shap.Explainer(lr.named_steps['clf'], X_test_trans)
            shap_values = explainer(X_test_trans)
            st.pyplot(shap.summary_plot(shap_values, X_test_trans, show=False, plot_type="bar"))

            # Explainable Boosting
            st.subheader("Explainable Boosting (EBM)")
            ebm = Pipeline([
                ('preproc', preproc),
                ('clf', ExplainableBoostingClassifier(random_state=42))
            ])
            ebm.fit(X_train, y_train)
            y_pred_ebm = ebm.predict(X_test)
            st.code(classification_report(y_test, y_pred_ebm), language='text')

            # LIME
            st.subheader("LIME Explanation")
            X_train_trans = lr.named_steps['preproc'].transform(X_train)
            lime_explainer = LimeTabularExplainer(
                training_data=np.array(X_train_trans),
                feature_names=lr.named_steps['preproc'].get_feature_names_out(),
                class_names=['Not Approved','Approved'],
                mode='classification'
            )
            instance_idx = st.slider("Select Instance for LIME", 0, len(X_test)-1, 0)
            X_instance = lr.named_steps['preproc'].transform(X_test.iloc[[instance_idx]])
            exp = lime_explainer.explain_instance(X_instance[0], lr.named_steps['clf'].predict_proba, num_features=10)
            st.pyplot(exp.as_pyplot_figure())

            # PDP
            st.subheader("Partial Dependence Display")
            pdp_features = ['Credit_Score','NETMONTHLYINCOME','time_since_recent_deliquency',
                            'CC_utilization','PL_enq_L6m','AGE','Tot_Missed_Pmnt','pct_active_tl']
            fig, ax = plt.subplots(figsize=(12,8))
            PartialDependenceDisplay.from_estimator(
                lr, X_train, features=[f for f in pdp_features if f in X_train.columns],
                kind='average', ax=ax
            )
            st.pyplot(fig)

        st.success("Model training and visualizations complete!")


Overwriting app.py


In [ ]:
!pip install streamlit pyngrok --quiet

from pyngrok import ngrok
!ngrok authtoken 2vtJQzYbRGamgeosVQrK8O4L84K_5N1twnJEvY5RgewsbSLGn
ngrok.kill()
!streamlit run app.py &>/dev/null &
import time
time.sleep(5)
public_url = ngrok.connect(addr=8501)
print("Streamlit app is live here:", public_url)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🌍 Streamlit app is live here: NgrokTunnel: "https://2ce12056f3bd.ngrok-free.app" -> "http://localhost:8501"
